In [ ]:
# Import Required Libraries
import traceback
import pandas as pd
import numpy as np
import json
from lxml import etree
import ast
import re
import uuid
import logging
from datetime import date, datetime, timedelta
from math import ceil
from concurrent.futures import ThreadPoolExecutor

import psycopg2
import psycopg2.extras
from psycopg2 import sql, DatabaseError
from psycopg2.extras import execute_values

import pymssql

# Setting up logging configuration
logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.StreamHandler()
    ]
)


# ============================================================================
# DATABASE CONNECTION & UTILITY FUNCTIONS
# ============================================================================

def initialize_database_connection(db_type, **kwargs):
    """
    Initializes and returns a database connection dynamically based on the provided database type.
    """
    try:
        if db_type.lower() == "postgres":
            connection = psycopg2.connect(
                dbname=kwargs["pg_dbname"],
                user=kwargs["pg_user"],
                password=kwargs["pg_password"],
                host=kwargs["pg_host"],
                port=kwargs["pg_port"],
            )

        elif db_type.lower() == "mssql":
            connection = pymssql.connect(
                server=kwargs["mssql_server"],
                user=kwargs["mssql_user"],
                password=kwargs["mssql_pass"],
                database=kwargs["mssql_db"],
                port=kwargs.get("mssql_port", 1433),
            )

        elif db_type.lower() == "mysql":
            connection = mysql.connector.connect(
                host=kwargs["host"],
                user=kwargs["user"],
                password=kwargs["password"],
                database=kwargs["database"],
                port=kwargs.get("port", 3306),
            )

        else:
            raise ValueError(f"Unsupported database type: {db_type}")

        return connection

    except Exception as e:
        logging.error(f"Failed to establish {db_type} connection: {e}")
        raise Exception(f"Failed establishing connection: {e}")


# ============================================================================
# GLOBAL BATCH AUDIT FUNCTIONS
# ============================================================================

def read_global_etl_batch_id(conn):
    """
    Reads the current running batch information from global batch audit table.
    
    Returns:
        tuple: (batch_run_id, run_end_datetime, load_type, run_frequency) or (None, None, None, None) if no running batch
        
    Raises:
        Exception: If error occurs while reading batch information
    """
    try:
        read_query = """
            SELECT * FROM silver_etl_master.silver_etl_global_batch_audit begba 
            WHERE begba.run_status = 'RUNNING'
            ORDER BY batch_run_id DESC, run_start_datetime DESC
        """
        
        batch_run_id = pd.read_sql_query(read_query, conn)
        
        if batch_run_id.empty:
            logging.info("No running batch found.")
            return None, None, None, None
        else:
            logging.info(f"Running batch found with batch_run_id: {batch_run_id.iloc[0]['batch_run_id']}")
            return (
                batch_run_id.iloc[0]["batch_run_id"], 
                batch_run_id.iloc[0]["run_start_datetime"],
                batch_run_id.iloc[0]["run_end_datetime"], 
                batch_run_id.iloc[0]["load_type"],
                batch_run_id.iloc[0]["run_frequency"]
            )
    except Exception as e:
        error_msg = f"Error while reading global ETL batch ID: {str(e)}"
        logging.error(error_msg)
        raise Exception(error_msg)


# ============================================================================
# CONFIG TABLE READING FUNCTIONS
# ============================================================================

def read_etl_config_table(
    connection,
    source_name,
    table_type,
    run_frequency,
    run_type,
    table_name=None,
):
    """
    Reads the parameter/configuration lookup table for ETL processing.
    """

    def safe_to_tuple(value):
        """Normalize input into a tuple."""
        if value is None:
            return tuple()
        if isinstance(value, (list, tuple)):
            return tuple(value)
        if isinstance(value, str):
            return (value,)
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, (list, tuple)):
                return tuple(parsed)
            return (parsed,)
        except (SyntaxError, ValueError):
            return (value,)

    try:
        # Normalize inputs to tuples
        source_name = safe_to_tuple(source_name)
        table_name = safe_to_tuple(table_name)
        table_type = safe_to_tuple(table_type)

        # ---- Source placeholders ----
        source_placeholders = ",".join(["%s"] * len(source_name))

        # ---- Table name condition ----
        table_name_condition = "TRUE"
        table_name_params = ()

        if len(table_name) > 0:
            table_placeholders = ",".join(["%s"] * len(table_name))
            table_name_condition = f"raw_table_name IN ({table_placeholders})"
            table_name_params = table_name

        # ---- Base Query ----
        query = f"""
            SELECT *
            FROM silver_etl_master.silver_etl_config_tbl sect
            WHERE is_active = 1
            AND source_name IN ({source_placeholders})
            AND table_type IN (%s)
            AND ({table_name_condition})
        """

        params = source_name + table_type + table_name_params

        # ---- Delta condition ----
        if run_type.lower() == "delta":
            query += " AND run_frequency = %s"
            params += (run_frequency,)

        # ---- Execute ----
        parameter_tbl = pd.read_sql_query(query, connection, params=params)
        return parameter_tbl    

    except Exception as e:
        logging.error(f"Error reading config: {e}")
        raise Exception(f"Failed to read config table: {e}")


# ============================================================================
# AUDIT TABLE FUNCTIONS
# ============================================================================

def update_audit_master_tbl(
    connection_dl_bronze,
    batch_run_id,
    etl_name,
    etl_start_time=None,
    etl_end_time=None,
    status=None,
    table_processed=0,
    table_succeeded=0,
    table_failed=0,
    error_msg=None,
    process_id=None,
    etl_schedule_time=None
):
    """
    Inserts or updates an ETL process record in the master audit table.
    
    Raises:
        Exception: If audit update fails
    """
    try:
        cursor = connection_dl_bronze.cursor()

        if status and status.lower() == "in-progress":
            if process_id == None:
                cursor.execute("""
                    SELECT COALESCE(MAX(master_audit_id), 0) + 1 AS process_id
                    FROM silver_etl_master.silver_etl_run_master_audit_tbl_test
                """)
                process_id = cursor.fetchone()[0]

            insert_query = """
                INSERT INTO silver_etl_master.silver_etl_run_master_audit_tbl_test (
                    master_audit_id,
                    etl_batch_id,
                    schedule_name,
                    schedule_timestamp,
                    status,
                    start_time,
                    end_time,
                    tables_processed,
                    tables_successful,
                    tables_failed,
                    error_message
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """
            cursor.execute(insert_query, (
                int(process_id),
                int(batch_run_id),
                etl_name,
                etl_schedule_time,
                status,
                etl_start_time,
                etl_end_time,
                table_processed,
                table_succeeded,
                table_failed,
                error_msg,
            ))
            logging.info(f"New Process Started with Process ID: {process_id}.")
        else:
            if process_id == None:
                raise Exception("Unable to process without Process ID")
            update_query = """
                UPDATE silver_etl_master.silver_etl_run_master_audit_tbl_test
                SET 
                    status = %s,
                    end_time = %s,
                    tables_processed = %s,
                    tables_successful = %s,
                    tables_failed = %s,
                    error_message = %s
                WHERE master_audit_id = %s
            """
            cursor.execute(update_query, (
                status,
                etl_end_time,
                table_processed,
                table_succeeded,
                table_failed,
                error_msg,
                process_id,
            ))
            logging.info(f"Process Completed for Process ID: {process_id}.")
        connection_dl_bronze.commit()
        return process_id
    except Exception as e:
        error_msg = f"Failed to update audit master table: {str(e)}"
        logging.error(error_msg)
        raise Exception(error_msg)


def upsert_etl_audit_record(
    connection_dl_bronze,
    process_id,
    batch_run_id,
    source_name,
    source_table,
    destination_table,
    operation,
    time_from,
    input_row,
    status,
    db_code='MB-101',
    time_to=None,
    output_row=0,
    error_code=None,
    result=None,
    etl_end_time=None,
    etl_start_time=None
):
    """
    Inserts or updates a record in the ETL audit table.
    
    Raises:
        Exception: If audit log operation fails
    """
    if db_code == None:
        db_code = 'MB-101'
    
    try:
        input_row = int(input_row) if input_row is not None else None
        output_row = int(output_row) if output_row is not None else None

        with connection_dl_bronze.cursor() as cur:
            if status.lower() == "in-progress":
                insert_to_audit_tbl_query = """
                    INSERT INTO silver_etl_master.silver_etl_run_table_audit_log (
                        process_id, batch_run_id, data_source, db_code, source_table_name, destination_table_name,
                        operation, time_from, time_to, etl_start_time, etl_end_time,
                        input_row, output_row, operation_result, errorcode, result
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                """
                cur.execute(insert_to_audit_tbl_query, (
                    int(process_id),
                    int(batch_run_id),
                    source_name,
                    db_code,
                    source_table,
                    destination_table,
                    operation,
                    time_from,
                    time_to,
                    etl_start_time,
                    time_to,
                    input_row,
                    output_row,
                    status,
                    error_code,
                    result,
                ))
                logging.info(f"ETL Process start logged successfully.")
            else:
                update_audit_tbl_query = """
                    UPDATE silver_etl_master.silver_etl_run_table_audit_log
                    SET time_to = %s,
                        etl_end_time = %s,
                        input_row = %s,
                        output_row = %s,
                        operation_result = %s,
                        errorcode = %s,
                        result = %s
                    WHERE process_id = %s AND batch_run_id = %s AND source_table_name = %s
                """
                cur.execute(update_audit_tbl_query, (
                    time_to,
                    time_to,
                    int(input_row),
                    int(output_row),
                    status,
                    error_code,
                    result,
                    int(process_id),
                    int(batch_run_id),
                    source_table,
                ))
                logging.info(f"ETL Process Completion logged successfully.")
            connection_dl_bronze.commit()
    except Exception as e:
        error_msg = f"Audit log failed: {str(e)}"
        logging.error(error_msg)
        connection_dl_bronze.rollback()
        raise Exception(error_msg)


# ============================================================================
# SCHEMA & TABLE MANAGEMENT FUNCTIONS
# ============================================================================

def get_json_and_array_columns(pg_conn, schema_name, table_name):
    """
    Fetches the names of columns in a PostgreSQL table that are of type JSON/JSONB or array.
    """
    json_columns = []
    array_columns = []

    query = """
        SELECT column_name, data_type, udt_name
        FROM information_schema.columns
        WHERE LOWER(table_schema) = LOWER(%s)
        AND LOWER(table_name) = LOWER(%s)
    """

    with pg_conn.cursor() as cursor:
        cursor.execute(query, (schema_name, table_name))
        for column_name, data_type, udt_name in cursor.fetchall():
            if data_type.lower() in ["json", "jsonb"]:
                json_columns.append(column_name)
            elif udt_name.startswith("_"):
                array_columns.append(column_name)

    return json_columns, array_columns


def get_xml_columns(pg_conn, schema_name, table_name):
    """
    Fetches the names of columns in a PostgreSQL table that are of type XML.
    """
    xml_columns = []

    query = """
        SELECT column_name
        FROM information_schema.columns
        WHERE LOWER(table_schema) = LOWER(%s)
        AND LOWER(table_name) = LOWER(%s)
        AND data_type = 'xml'
    """

    with pg_conn.cursor() as cursor:
        cursor.execute(query, (schema_name, table_name))
        for row in cursor.fetchall():
            xml_columns.append(row[0])

    return xml_columns


def get_table_columns(conn, table_name, schema='public'):
    """Get table columns."""
    query = """
    SELECT column_name 
    FROM information_schema.columns 
    WHERE table_schema = %s AND table_name = %s;
    """
    with conn.cursor() as cur:
        cur.execute(query, (schema, table_name))
        result = cur.fetchall()
    return [row[0] for row in result]


def get_common_columns(df, table_columns):
    """Get common columns between df and db table."""
    return [col for col in df.columns if col in table_columns]


def check_table_exists_and_get_columns(conn, schema, table_name):
    """
    Checks if a table exists in the specified schema and retrieves its column names.
    """
    table_exists_query = """
        SELECT EXISTS (
            SELECT 1
            FROM information_schema.tables 
            WHERE table_schema = %s 
            AND table_name = %s
        );
    """

    column_names_query = """
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = %s
        AND table_name = %s
        ORDER BY ordinal_position;
    """

    try:
        with conn.cursor() as cursor:
            cursor.execute(table_exists_query, (schema, table_name))
            exists = cursor.fetchone()[0]

            if exists:
                cursor.execute(column_names_query, (schema, table_name))
                columns = [row[0] for row in cursor.fetchall()]
                return exists, columns
            else:
                return exists, []

    except Exception as e:
        raise RuntimeError(f"Error checking table or fetching columns: {e}")


def sync_table_columns_with_target(
    conn, schema, table_name, existing_columns, target_columns
):
    """
    Syncs a table to match the exact target columns.
    """
    try:
        with conn.cursor() as cursor:
            metadata_query = """
                SELECT column_name, data_type
                FROM information_schema.columns
                WHERE table_schema = %s AND table_name = %s;
            """
            cursor.execute(metadata_query, (schema, table_name))
            column_metadata = dict(cursor.fetchall())

            existing_set = set(existing_columns)
            target_set = set(target_columns)

            missing_columns = target_set - existing_set
            extra_columns = existing_set - target_set

            # Add missing target columns
            for col in missing_columns:
                alter_add = f'ALTER TABLE "{schema}"."{table_name}" ADD COLUMN "{col}" TEXT;'
                cursor.execute(alter_add)

            # Handle insertion_date separately
            insertion_date_exists = "insertion_date" in column_metadata
            insertion_date_valid = insertion_date_exists and column_metadata[
                "insertion_date"
            ].lower() in ["timestamp", "timestamp without time zone"]

            # Remove insertion_date from extra_columns if it's valid
            if insertion_date_valid:
                extra_columns.discard("insertion_date")

            # Drop extra columns (excluding valid insertion_date)
            for col in extra_columns:
                alter_drop = f'ALTER TABLE "{schema}"."{table_name}" DROP COLUMN "{col}";'
                cursor.execute(alter_drop)

            # Drop and recreate insertion_date if it's not valid
            if not insertion_date_valid:
                cursor.execute(
                    f"""
                    ALTER TABLE "{schema}"."{table_name}" 
                    ADD COLUMN "insertion_date" TIMESTAMP DEFAULT CURRENT_TIMESTAMP;
                """
                )

            conn.commit()

    except Exception as e:
        conn.rollback()
        raise RuntimeError(f"Error syncing table columns: {e}")


def get_filtered_schema_for_create(
    conn, bronze_table, selected_columns, schema="public"
):
    """
    Fetch filtered schema from a PostgreSQL table, formatted for CREATE TABLE.
    """
    query = """
        SELECT 
            column_name,
            data_type,
            udt_name,
            character_maximum_length,
            numeric_precision,
            numeric_scale,
            is_nullable,
            is_identity
        FROM information_schema.columns
        WHERE table_schema = %s AND table_name = %s
        ORDER BY ordinal_position;
    """

    try:
        with conn.cursor() as cursor:
            cursor.execute(query, (schema, bronze_table))
            rows = cursor.fetchall()

        df = pd.DataFrame(
            rows,
            columns=[
                "column_name",
                "data_type",
                "udt_name",
                "char_length",
                "num_precision",
                "num_scale",
                "is_nullable",
                "is_identity",
            ],
        )

        column_defs = []
        found_cols = set(df["column_name"].tolist())

        for col in selected_columns:
            if col in found_cols:
                row = df[df["column_name"] == col].iloc[0]
                udt = row["udt_name"]
                dbtype = row["data_type"]
                nullable = "NULL" if row["is_nullable"] == "YES" else "NOT NULL"

                # Handle identity columns (serial-like)
                if row["is_identity"] == "YES":
                    column_defs.append(f"{col} GENERATED ALWAYS AS IDENTITY")
                    continue

                # Handle varchar
                if udt in ("varchar", "bpchar") and pd.notnull(row["char_length"]):
                    dbtype_str = f"VARCHAR({int(row['char_length'])})"
                elif (
                    udt == "numeric"
                    and pd.notnull(row["num_precision"])
                    and pd.notnull(row["num_scale"])
                ):
                    dbtype_str = (
                        f"NUMERIC({int(row['num_precision'])}, {int(row['num_scale'])})"
                    )
                elif udt == "int2":
                    dbtype_str = "SMALLINT"
                elif udt == "int4":
                    dbtype_str = "INTEGER"
                elif udt == "int8":
                    dbtype_str = "BIGINT"
                elif udt == "float4":
                    dbtype_str = "REAL"
                elif udt == "float8":
                    dbtype_str = "DOUBLE PRECISION"
                elif udt == "bool":
                    dbtype_str = "BOOLEAN"
                elif udt == "uuid":
                    dbtype_str = "UUID"
                elif udt == "text":
                    dbtype_str = "TEXT"
                elif udt == "json":
                    dbtype_str = "JSON"
                elif udt == "jsonb":
                    dbtype_str = "JSONB"
                elif udt == "bytea":
                    dbtype_str = "BYTEA"
                elif udt.startswith("_"):
                    base_type = udt[1:]
                    dbtype_str = f"{base_type.upper()}[]"
                elif "timestamp" in dbtype:
                    dbtype_str = dbtype.upper()
                elif "date" in dbtype:
                    dbtype_str = "DATE"
                elif "time" in dbtype:
                    dbtype_str = dbtype.upper()
                elif "interval" in dbtype:
                    dbtype_str = "INTERVAL"
                elif dbtype == "USER-DEFINED":
                    dbtype_str = udt.upper()
                else:
                    dbtype_str = dbtype.upper()

                column_defs.append(f'"{col}" {dbtype_str} {nullable}')
            else:
                # If the column is missing, define as TEXT and nullable
                column_defs.append(f'"{col}" TEXT NULL')

        return column_defs

    except Exception as e:
        raise RuntimeError(f"Error fetching schema: {e}")


def create_table_in_silver(conn, schema, target_table, column_definitions, load_key=None):
    """
    Create a table in the Silver layer using provided column definitions.
    """
    try:
        with conn.cursor() as cursor:
            quoted_schema = f'"{schema}"'
            quoted_table = f'"{target_table}"'
            
            pk_columns=None
            if isinstance(load_key, str):
                pk_columns = [col.strip() for col in load_key.split(",") if col.strip()]
            elif load_key is None:
                pk_columns = None
            elif not isinstance(load_key, (list, tuple)):
                raise ValueError("load_key must be a string or list/tuple of column names.")
            
            # Quote each column name to avoid conflicts with reserved words       
            if pk_columns:
                quoted_pk_columns = ', '.join(f'"{col}"' for col in pk_columns)
                pk_clause = f", PRIMARY KEY ({quoted_pk_columns})"
            else:
                pk_clause = ""
            
            # Add the insertion_date column first
            insertion_column = "insertion_date TIMESTAMP DEFAULT NOW()"
            all_columns = column_definitions + [insertion_column]
            
            columns_sql = ",\n".join(all_columns) + pk_clause
            create_sql = f"""
            CREATE TABLE IF NOT EXISTS {quoted_schema}.{quoted_table} (
                {columns_sql});
            """

            cursor.execute(create_sql)
            conn.commit()
            return f"Table {schema}.{target_table} created successfully"
    except Exception as e:
        conn.rollback()
        return f"Error creating table: {e}"


# ============================================================================
# DATA EXTRACTION FUNCTIONS
# ============================================================================

def build_incremental_where_clause(
    extract_key_str, last_processed_date, new_processed_date=None, dbtype=None
):
    """
    Builds a SQL WHERE clause for incremental extraction based on extract key columns and date boundaries.
    """
    if not extract_key_str or not last_processed_date:
        return ""

    columns = [col.strip() for col in extract_key_str.split(",") if col.strip()]
    if not columns:
        return ""

    def format_date_for_db(date_value, db_type):
        if not date_value:
            return None
        
        if hasattr(date_value, 'strftime'):
            date_string = date_value.strftime('%Y-%m-%d %H:%M:%S.%f') if hasattr(date_value, 'microsecond') and date_value.microsecond else date_value.strftime('%Y-%m-%d %H:%M:%S')
        else:
            date_string = str(date_value)
        
        if db_type and db_type.lower() in ('mssql', 'mysql'):
            if '.' in date_string:
                date_part, time_part = date_string.split('.', 1)
                time_part = time_part[:3] if len(time_part) > 3 else time_part
                date_string = f"{date_part}.{time_part}"
            return f"CAST('{date_string}' AS DATETIME)"
        elif db_type and db_type.lower() == 'postgres':
            return f"'{date_string}'::timestamp"
        else:
            return f"'{date_string}'"

    date_lower = format_date_for_db(last_processed_date, dbtype)
    date_upper = format_date_for_db(new_processed_date, dbtype) if new_processed_date else None

    conditions = []
    for col in columns:
        if date_upper:
            condition = f'("{col}" > {date_lower} AND "{col}" < {date_upper})'
        else:
            condition = f'("{col}" > {date_lower})'
        conditions.append(condition)

    where_clause = "WHERE " + " OR ".join(conditions)
    return where_clause


def build_upper_bound_where_clause(extract_key_str, upper_bound_date, dbtype=None):
    """
    Builds a SQL WHERE clause for FULL load with upper bound filtering.
    """
    if not extract_key_str or not upper_bound_date:
        return ""

    columns = [col.strip() for col in extract_key_str.split(",") if col.strip()]
    if not columns:
        return ""

    def format_date_for_db(date_value, db_type):
        if not date_value:
            return None
        
        if hasattr(date_value, 'strftime'):
            date_string = date_value.strftime('%Y-%m-%d %H:%M:%S.%f') if hasattr(date_value, 'microsecond') and date_value.microsecond else date_value.strftime('%Y-%m-%d %H:%M:%S')
        else:
            date_string = str(date_value)
        
        if db_type and db_type.lower() in ('mssql', 'mysql'):
            if '.' in date_string:
                date_part, time_part = date_string.split('.', 1)
                time_part = time_part[:3] if len(time_part) > 3 else time_part
                date_string = f"{date_part}.{time_part}"
            return f"CAST('{date_string}' AS DATETIME)"
        elif db_type and db_type.lower() == 'postgres':
            return f"'{date_string}'::timestamp"
        else:
            return f"'{date_string}'"

    date_upper = format_date_for_db(upper_bound_date, dbtype)

    conditions = []
    for col in columns:
        condition = f'("{col}" < {date_upper})'
        conditions.append(condition)

    where_clause = "WHERE " + " OR ".join(conditions)
    return where_clause


def extract_source_table_data(df_row, connection, last_processed_date=None, batch_size=100000, where_clause=None):
    """Extracts data from a source table in batches."""
    source_schema = df_row.get("raw_schema_name", "").strip()
    source_table = df_row.get("raw_table_name", "").strip()
    db_type = df_row.get("db_connection_type", "").strip().lower()
    extract_key = (df_row.get("extract_key", "") or "").strip()
    load_key = (df_row.get("load_key", "") or "").strip()
    operation = df_row.get("load_type", "").strip()

    if not source_table:
        raise ValueError("Source table name is missing")

    table_name_for_query = f"[{source_table}]" if source_table.lower() == "user" and db_type == "mssql" else source_table
    table_reference = f'''"{source_schema}"."{table_name_for_query}"''' if source_schema else table_name_for_query

    order_columns = [col.strip() for col in (load_key or extract_key).split(",") if col.strip()]
    if not order_columns:
        raise ValueError("Both load_key and extract_key are missing")
    logging.info(f"Using order columns: {order_columns}") # ----------> Debugging log

    where_clause_str = ""
    if operation.lower() != "full" and where_clause:
        where_clause_str = where_clause.strip()
        if not where_clause_str.lower().startswith("where"):
            where_clause_str = f"WHERE {where_clause_str}"

    tables_requiring_casting = {"Organization", "PrivateFare", "TravelerProfile"}
    use_custom_select = source_table in tables_requiring_casting

    select_columns = "*"
    end_date_expr = ""
    if use_custom_select:
        if db_type == "postgres":
            column_query = f"""
                SELECT column_name FROM information_schema.columns
                WHERE table_schema = %s AND table_name = %s AND column_name != 'EndDate'
            """
            with connection.cursor(cursor_factory=psycopg2.extras.DictCursor) as cursor:
                cursor.execute(column_query, (source_schema, source_table))
                columns = [row["column_name"] for row in cursor.fetchall()]
        elif db_type == "mssql":
            column_query = f"""
                SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS
                WHERE TABLE_SCHEMA = %s AND TABLE_NAME = %s AND COLUMN_NAME != 'EndDate'
            """
            with connection.cursor(as_dict=True) as cursor:
                cursor.execute(query, (source_schema, source_table))
                columns = [row["COLUMN_NAME"] for row in cursor.fetchall()]
        elif db_type == "mysql":
            column_query = f"""
                SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS
                WHERE TABLE_SCHEMA = %s AND TABLE_NAME = %s AND COLUMN_NAME != 'EndDate'
            """
            with connection.cursor(dictionary=True) as cursor:
                cursor.execute(column_query, (source_schema, source_table))
                columns = [row["COLUMN_NAME"] for row in cursor.fetchall()]
        else:
            raise ValueError(f"Unsupported DB type: {db_type}")

        select_columns = ", ".join(columns)
        if db_type == "mssql":
            end_date_expr = "CASE WHEN EndDate >= '9999-12-31 23:59:59.000' THEN CAST('9999-12-31 22:59:59.000' AS DATETIME2) ELSE EndDate END AS EndDate"
        elif db_type == "postgres":
            end_date_expr = "CASE WHEN \"EndDate\" >= '9999-12-31 23:59:59'::timestamp THEN '9999-12-31 22:59:59'::timestamp ELSE \"EndDate\" END AS \"EndDate\""
        else:
            end_date_expr = "CASE WHEN EndDate >= '9999-12-31 23:59:59' THEN CAST('9999-12-31 22:59:59' AS DATETIME) ELSE EndDate END AS EndDate"
        select_columns = f"{select_columns}, {end_date_expr}"

    offset = 0
    batch_count = 0

    while True:
        if db_type == "postgres":
            order_by = ", ".join([f'"{col}"' for col in order_columns])
        elif db_type == "mysql":
            order_by = ", ".join([f'`{col}`' for col in order_columns])
        else:
            order_by = ", ".join(order_columns)

        if use_custom_select:
            select_part = select_columns
        else:
            select_part = "*"

        if db_type == "postgres":
            query = f"SELECT {select_part} FROM {table_reference} {where_clause_str} ORDER BY {order_by} LIMIT {batch_size} OFFSET {offset}"
        elif db_type == "mysql":
            query = f"SELECT {select_part} FROM {table_reference} {where_clause_str} ORDER BY {order_by} LIMIT {batch_size} OFFSET {offset}"
        elif db_type == "mssql":
            query = f"SELECT {select_part} FROM {table_reference} {where_clause_str} ORDER BY {order_by} OFFSET {offset} ROWS FETCH NEXT {batch_size} ROWS ONLY"
        else:
            raise ValueError(f"Unsupported DB type: {db_type}")

        df_chunk = pd.read_sql_query(query, connection)
        if df_chunk.empty:
            break

        batch_count += 1
        yield df_chunk
        offset += len(df_chunk)
        if len(df_chunk) < batch_size:
            break


# ============================================================================
# DATA CLEANING & WRITING FUNCTIONS
# ============================================================================

def ensure_uuid_strings_auto(df: pd.DataFrame) -> pd.DataFrame:
    """
    Auto-detect UUID values and convert them to uppercase strings.
    """
    uuid_regex = re.compile(
        r"^[0-9a-fA-F]{8}-"
        r"[0-9a-fA-F]{4}-"
        r"[1-5][0-9a-fA-F]{3}-"
        r"[89abAB][0-9a-fA-F]{3}-"
        r"[0-9a-fA-F]{12}$"
    )

    def convert(val):
        if isinstance(val, uuid.UUID):
            return str(val).upper()
        elif isinstance(val, str):
            val = val.strip()
            if uuid_regex.match(val):
                try:
                    return str(uuid.UUID(val)).upper()
                except Exception:
                    return val
        return val

    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].apply(convert)

    return df


def clean_dataframe_for_db(df: pd.DataFrame, uuid_columns=None) -> pd.DataFrame:
    """
    Cleans the DataFrame for safe database insertion.
    """
    df_cleaned = df.copy()
    for col in df_cleaned.columns:
        if df_cleaned[col].dtype == object or pd.api.types.is_string_dtype(df_cleaned[col]):
            df_cleaned[col] = df_cleaned[col].apply(
                lambda x: (
                    None
                    if isinstance(x, str) and ("\x00" in x or x.strip() == "")
                    else x
                )
            )

    df_cleaned = df_cleaned.where(pd.notnull(df_cleaned), None)
    df_cleaned = ensure_uuid_strings_auto(df_cleaned)
    return df_cleaned


def write_batch_to_datalake(
    pg_conn,
    batch_df,
    target_schema,
    target_table,
    load_type,
    conflict_columns=None,
    json_columns=None,
    array_columns=None,
    xml_columns=None,        # ' ADDED: XML column support
):
    """
    Writes a batch of data from a DataFrame to a PostgreSQL table in the datalake.
    """
    if batch_df.empty:
        return True

    # Step 1: Add insertion_date if missing
    if "insertion_date" not in batch_df.columns:
        batch_df["insertion_date"] = date.today()

    # Step 2: Replace NaN/NaT with None
    cleaned_df = batch_df.where(pd.notnull(batch_df), None)
    cleaned_df = cleaned_df.replace({pd.NaT: None, pd.NA: None, float("nan"): None})

    # Step 3: Serialization setup
    json_columns = json_columns or []
    array_columns = array_columns or []
    xml_columns = xml_columns or []        # ' ADDED

    # '' XML Cleaning Helper ''''''''''''''''''''''''''''''''''''''''''''''''''
    def clean_xml_value(value):
        if value is None:
            return None
        if not isinstance(value, str):
            return value
        try:
            # Try parsing directly
            etree.fromstring(value.encode("utf-8"))
            return value
        except Exception:
            try:
                # Fix common invalid entities
                value_fixed = value.replace("&Lt;", "&lt;")
                value_fixed = value_fixed.replace("&Gt;", "&gt;")
                # Fix unescaped ampersands
                value_fixed = re.sub(
                    r'&(?!amp;|lt;|gt;|apos;|quot;)',
                    '&amp;',
                    value_fixed
                )
                etree.fromstring(value_fixed.encode("utf-8"))
                return value_fixed
            except Exception:
                # Return None to prevent batch crash on unfixable XML
                return None
    # ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''

    def serialize_cell(val, is_json=False, is_array=False):
        try:
            if is_json:
                return json.dumps(val, default=str)
            if is_array:
                return (
                    f'{{{",".join(map(str, val))}}}' if isinstance(val, list) else None
                )
        except Exception:
            return None
        return val

    for col in cleaned_df.columns:
        if col in json_columns:
            cleaned_df[col] = cleaned_df[col].apply(
                lambda x: serialize_cell(x, is_json=True)
            )
        elif col in array_columns:
            cleaned_df[col] = cleaned_df[col].apply(
                lambda x: serialize_cell(x, is_array=True)
            )
        elif col in xml_columns:                    # ' ADDED: apply XML cleaner
            cleaned_df[col] = cleaned_df[col].apply(clean_xml_value)

    # Step 4: Prepare insert query
    columns = list(cleaned_df.columns)
    values = [tuple(row) for row in cleaned_df.to_numpy()]
    columns_quoted = ", ".join(f'"{col}"' for col in columns)

    if load_type.lower() in ["append", "full"]:
        insert_query = f"""
            INSERT INTO "{target_schema}"."{target_table}" ({columns_quoted})
            VALUES %s
        """
    elif load_type.lower() == "incremental":
        if not conflict_columns or not isinstance(conflict_columns, list):
            logging.error("'conflict_columns' must be provided for incremental loads.")
            return False
        conflict_cols_str = ", ".join(f'"{col}"' for col in conflict_columns)
        update_set = ", ".join(
            f'"{col}" = EXCLUDED."{col}"'
            for col in columns
            if col not in conflict_columns
        )
        insert_query = f"""
            INSERT INTO "{target_schema}"."{target_table}" ({columns_quoted})
            VALUES %s
            ON CONFLICT ({conflict_cols_str})
            DO UPDATE SET {update_set}
        """
    else:
        logging.error(f"Unknown load_type: {load_type}")
        return False

    # Step 5: Execute insert
    try:
        with pg_conn.cursor() as cursor:
            execute_values(cursor, insert_query, values)
            pg_conn.commit()
            return True
    except Exception as e:
        pg_conn.rollback()
        logging.error(f"Batch insert failed: {e}")
        return False


# ============================================================================
# TRANSFORMATION FUNCTIONS
# ============================================================================

def apply_case_unification(df, config, process_id, source_name, table_name, conn):
    """Apply case unification on DataFrame columns based on provided config."""
    try:
        case_map = {
            "UPPER": lambda x: x.upper() if isinstance(x, str) else x,
            "LOWER": lambda x: x.lower() if isinstance(x, str) else x,
            "TITLE": lambda x: x.title() if isinstance(x, str) else x,
            "CAPITALIZE": lambda x: x.capitalize() if isinstance(x, str) else x,
            "STRIP": lambda x: x.strip() if isinstance(x, str) else x,
            "SENTENCE": lambda x: x.capitalize() if isinstance(x, str) else x
        }

        for col, case_type in config.items():
            if col in df.columns:
                case_type = case_type.upper()
                if case_type in case_map:
                    df[col] = df[col].apply(case_map[case_type])

        return df

    except Exception as e:
        logging.error(f"Case unification error: {str(e)}")
        raise


def apply_datetime_formatting(df, columns_config, process_id, source_name, table_name, conn):
    """Applies datetime formatting to specified columns."""
    if not columns_config:
        return df

    try:
        if isinstance(columns_config, str):
            format_config = json.loads(columns_config)
        else:
            format_config = columns_config

        format_mapping = {
            "YYYY-MM-DD": "%Y-%m-%d",
            "YYYY-MM-DD HH:mm:ss": "%Y-%m-%d %H:%M:%S",
            "MM/DD/YYYY": "%m/%d/%Y",
            "DD-MM-YYYY": "%d-%m-%Y",
        }

        for col, user_format in format_config.items():
            if col in df.columns:
                pandas_format = format_mapping.get(user_format, "%Y-%m-%d")
                converted = pd.to_datetime(df[col], errors="coerce", format=pandas_format)
                df.loc[:, col] = converted
                df.loc[:, col] = df[col].dt.strftime(pandas_format)

    except Exception as e:
        logging.error(f"Datetime formatting error: {str(e)}")
        raise

    return df


def remove_duplicates(df, process_id, source_name, table_name, conn):
    """Removes duplicate rows from a DataFrame."""
    try:
        return df.drop_duplicates()
    except Exception as e:
        logging.error(f"Duplicate removal error: {str(e)}")
        raise


def apply_post_2015_filter(df, columns_config, process_id, source_name, table_name, conn):
    """Filters rows where date columns have date >= 2015-01-01."""
    if not columns_config:
        return df

    try:
        if isinstance(columns_config, str):
            try:
                columns = json.loads(columns_config)
                if not isinstance(columns, list):
                    raise ValueError("columns_config JSON must be a list")
            except json.JSONDecodeError:
                columns = [col.strip() for col in columns_config.split(",")]
        else:
            columns = columns_config

        mask = pd.Series(False, index=df.index)
        
        for col in columns:
            if col in df.columns:
                col_dates = pd.to_datetime(df[col], errors="coerce")
                
                if hasattr(col_dates.dtype, 'tz') and col_dates.dtype.tz is not None:
                    comparison_date = pd.Timestamp("2015-01-01", tz='UTC')
                    col_mask = col_dates >= comparison_date
                else:
                    comparison_date = pd.Timestamp("2015-01-01")
                    col_mask = col_dates >= comparison_date
                
                mask = mask | col_mask

        return df[mask]

    except Exception as e:
        logging.error(f"Post-2015 filter error: {str(e)}")
        raise


def flatten_json_xml(df, config):
    """Flattens JSON/XML columns in DataFrame."""
    if not config:
        return df

    try:
        if isinstance(config, str):
            config = json.loads(config)

        for column, column_config in config.items():
            if column not in df.columns:
                continue

            extract_all = column_config.get("extract_all", False)
            raw_keys = column_config.get("keys_to_extract", []) if not extract_all else []
            keys_to_extract = []
            alias_map = {}

            for item in raw_keys:
                if isinstance(item, dict):
                    path = item.get("path")
                    alias = item.get("alias", path)
                    keys_to_extract.append(path)
                    alias_map[path] = alias
                else:
                    keys_to_extract.append(item)
                    alias_map[item] = f"{column}_{item.replace('/', '_').replace('@', 'attr_')}"

            extracted_data = df[column].apply(
                lambda x: extract_keys(x, extract_all, keys_to_extract, alias_map)
            )
            extracted_df = pd.DataFrame(extracted_data.tolist())
            df = pd.concat([df, extracted_df], axis=1)

    except Exception as e:
        logging.error(f"Error in flatten_json_xml: {str(e)}")

    return df


def extract_keys(data, extract_all, keys_to_extract, alias_map=None):
    if not data:
        return {}

    try:
        if isinstance(data, dict):
            data_dict = data
        elif isinstance(data, str) and data.strip().startswith(('{', '[')):
            data_dict = json.loads(data)
        elif isinstance(data, str) and data.strip().startswith('<'):
            root = etree.fromstring(data.encode())
            result = {}

            if extract_all:
                for elem in root.iter():
                    tag = etree.QName(elem).localname
                    if elem.text and elem.text.strip():
                        result[tag] = elem.text.strip()
                    for attr, val in elem.attrib.items():
                        result[f"{tag}_attr_{attr}"] = val
            else:
                for key in keys_to_extract:
                    alias = alias_map.get(key, key)
                    if '/@' in key:
                        path, attr = key.rsplit('/@', 1)
                        nodes = root.xpath(f'//{path}')
                        result[alias] = nodes[0].get(attr) if nodes and attr in nodes[0].attrib else None
                    else:
                        value = root.xpath(f'//{key}/text()')
                        result[alias] = value[0] if value else None

            return result

        else:
            return {}

        if extract_all:
            return data_dict

        return {
            alias_map.get(k, k): get_json_nested_value(data_dict, k)
            for k in keys_to_extract
        }

    except Exception:
        return {}


def get_json_nested_value(data_dict, path):
    try:
        keys = path.replace('.', '/').split('/')
        for k in keys:
            if isinstance(data_dict, dict):
                data_dict = data_dict.get(k)
            else:
                return None
        return data_dict
    except:
        return None


def get_new_columns_from_config(config_json):
    """Extract new column names from indicator and join enrichment configs"""
    new_columns = []

    if not config_json:
        return new_columns

    try:
        if isinstance(config_json, str):
            config = json.loads(config_json)
        else:
            config = config_json

        if isinstance(config, dict):
            for indicator_config in config.values():
                if "new_column_name" in indicator_config:
                    new_columns.append(indicator_config["new_column_name"])

        if "joins" in config:
            for join in config["joins"]:
                if "selected_columns" in join:
                    for col in join["selected_columns"]:
                        new_columns.append(col.get("alias") or col.get("column"))

    except Exception as e:
        logging.error(f"Error parsing config for new columns: {str(e)}")

    return [col for col in new_columns if col]


def unify_country_code(df, columns):
    """Unifies country codes in specified columns."""
    country_mapping = {"US": "USA", "IN": "IND", "GB": "UK"}
    if isinstance(columns, str):
        columns = [col.strip() for col in columns.split(",")]

    for col in columns:
        if col in df.columns:
            df[col] = df[col].map(country_mapping).fillna(df[col])
    return df


def handle_null_values(df, process_id, source_name, table_name, conn):
    """Replace all empty strings with None."""
    try:
        return df.replace("", None)
    except Exception as e:
        logging.error(f"Null value handling error: {str(e)}")
        raise


def index_partitioning(df):
    """Adds a sequential partition index column."""
    df["partition_index"] = range(1, len(df) + 1)
    return df


def trim_strings(df):
    """Trim, normalize spaces, and INITCAP string columns."""
    
    def clean_value(x):
        if isinstance(x, str):
            x = x.strip()
            x = re.sub(r'\s+', ' ', x)
            x = x.title()
        return x

    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].apply(clean_value)

    return df


def standardize_currency(df, columns):
    """Standardizes currency values."""
    currency_conversion = {"USD": 1, "INR": 0.012, "EUR": 1.1}
    for col in columns:
        if col in df.columns:
            df[col] = df[col] * df[f"{col}_currency"].map(currency_conversion).fillna(1)
    return df


def enrich_geographic_data(df, latitude_col, longitude_col):
    """Enriches geographic data."""
    df[latitude_col].fillna(40.7128, inplace=True)
    df[longitude_col].fillna(-74.0060, inplace=True)
    return df


def standardize_base_currency(df, columns):
    """Standardizes to base currency."""
    for col in columns:
        if col in df.columns:
            df[col] = df[col] * 1.2
    return df


def apply_value_mapping_unification(df, config):
    """Unify values using mapping."""
    if not config:
        return df

    try:
        if isinstance(config, str):
            mapping_config = json.loads(config)
        else:
            mapping_config = config

        for col, value_map in mapping_config.items():
            if col in df.columns:
                reverse_mapping = {}
                for target_val, source_vals in value_map.items():
                    for source_val in source_vals:
                        reverse_mapping[source_val] = target_val
                df[col] = df[col].replace(reverse_mapping)

    except Exception as e:
        logging.error(f"Value mapping unification error: {str(e)}")
        raise

    return df


# ============================================================================
# TABLE PROCESSING FUNCTIONS
# ============================================================================

def process_table_delta(
    connection_silver_db,
    connection_raw_db,
    row,
    batch_size,
    process_id,
    batch_run_id,
    last_processed_date_val=None,
    etl_start_datetime=None,
    dbtype='postgres'
):
    """Process a table from raw to silver layer - DELTA/INCREMENTAL load."""
    final_status = 'Failed'
    total_fetched = 0
    total_processed = 0
    error_message = None
    
    source_schema_name = row['raw_schema_name']
    source_table_name = row['raw_table_name']
    load_type = row['load_type']
    target_schema_name = row['clean_schema_name']
    target_table_name = row['clean_table_name']
    target_columns = [col.strip() for col in row['column_name'].split(',')]
    extract_key = row['extract_key']
    load_key = row['load_key']
    source_name = row['source_name']
    
    try:
        # Build where clause for delta load
        where_clause = build_incremental_where_clause(
            extract_key_str=extract_key,
            last_processed_date=last_processed_date_val,
            new_processed_date=etl_start_datetime, 
            dbtype=dbtype
        )
        logging.info(f"Where clause: {where_clause}")
        
        # Delete existing data in the time range for append mode
        if load_type.lower() == 'append' and last_processed_date_val:
            delete_query = f'DELETE FROM "{target_schema_name}"."{target_table_name}" {where_clause};'
            with connection_silver_db.cursor() as cursor:
                cursor.execute(delete_query)
                connection_silver_db.commit()
        
        conflict_columns = [col.strip() for col in load_key.split(',')] if load_key else []
        table_columns = get_table_columns(connection_silver_db, target_table_name, target_schema_name)
        json_cols, array_cols = get_json_and_array_columns(connection_silver_db, target_schema_name, target_table_name)
        xml_cols = get_xml_columns(connection_silver_db, target_schema_name, target_table_name)   # ' ADDED
        new_columns_to_check = []
        batch_count = 0
        
        # Extract data from raw layer in batches
        for batch in extract_source_table_data(row, connection_raw_db, last_processed_date=last_processed_date_val, where_clause=where_clause, batch_size=batch_size): 
            batch_count += 1
            input_data_count = len(batch)
            
            # Apply Transformations
            if row.get('case_unification'):
                batch = apply_case_unification(batch, row['case_unification'], process_id, source_name, row['raw_table_name'], connection_silver_db)

            if row.get('is_datetime_formatting'):
                batch = apply_datetime_formatting(batch, row['is_datetime_formatting'], process_id, source_name, row['raw_table_name'], connection_silver_db)

            if row.get('is_post_2015_filter') == '1':
                batch = apply_post_2015_filter(batch, row['extract_key'], process_id, source_name, row['raw_table_name'], connection_silver_db)

            if row.get('is_json_xml_flatten'):
                batch = flatten_json_xml(batch, row['is_json_xml_flatten'])

            if row.get('is_add_indicator'):
                new_columns_to_check.extend(get_new_columns_from_config(row['is_add_indicator']))
                new_columns_to_check = list(set(new_columns_to_check))
                for col in new_columns_to_check:
                    if col not in batch.columns:
                        batch[col] = None

            if row.get('is_join_enrichment'):
                new_columns_to_check.extend(get_new_columns_from_config(row['is_join_enrichment']))
                new_columns_to_check = list(set(new_columns_to_check))
                for col in new_columns_to_check:
                    if col not in batch.columns:
                        batch[col] = None

            if row.get('is_country_code_unification'):
                batch = unify_country_code(batch, row['is_country_code_unification'])

            if row.get('is_null_handling') == '1':
                batch = handle_null_values(batch, process_id, source_name, row['raw_table_name'], connection_silver_db)

            if row.get('is_index_partition'):
                batch = index_partitioning(batch)

            if row.get('is_string_trimming') == '1':
                batch = trim_strings(batch)

            if row.get('is_currency_standardization'):
                batch = standardize_currency(batch, row['is_currency_standardization'].split(", "))

            if row.get('geographic_enrichment'):
                batch = enrich_geographic_data(batch, "latitude", "longitude")

            if row.get('base_currency') == 1:
                batch = standardize_base_currency(batch, row['column_name'].split(", "))

            if row.get('value_mapping_unification'):
                batch = apply_value_mapping_unification(batch, row['value_mapping_unification'])

            if row.get('is_duplicate_check') == '1':
                batch = remove_duplicates(batch, process_id, source_name, row["raw_table_name"], connection_silver_db)
                
            total_fetched += len(batch)
            common_columns = get_common_columns(batch, table_columns)
            filtered_batch = batch[common_columns]
            filtered_batch['insertion_date'] = datetime.now()

            is_writing_success = write_batch_to_datalake(
                pg_conn=connection_silver_db,
                batch_df=filtered_batch,
                target_schema=target_schema_name,
                target_table=target_table_name,
                json_columns=json_cols,
                array_columns=array_cols,
                xml_columns=xml_cols,           # ' ADDED
                load_type=load_type,
                conflict_columns=conflict_columns
            )

            if is_writing_success:
                total_processed += len(filtered_batch)

        # Determine final status
        if total_fetched == total_processed and total_fetched > 0:
            final_status = 'Success'
        elif total_processed > 0:
            final_status = 'Partial Success'
            error_message = f"Processed {total_processed} out of {total_fetched} records"
        elif total_fetched == 0 and total_processed == 0:
            final_status = 'Success'
        else:
            final_status = 'Failed'
            error_message = f"Row count mismatch: Fetched={total_fetched}, Processed={total_processed}"
    
    except Exception as e:
        error_message = str(e)
        final_status = 'Failed'
        
    return total_fetched, total_processed, final_status, error_message


def process_table_full(
    connection_silver_db,
    connection_raw_db,
    process_id,
    row,
    upper_bound_time,
    batch_size=100000,
    dbtype='postgres'
):
    """Process a table from raw to silver layer - FULL load with upper bound time filtering."""
    final_status = "Failed"
    error_message = None
    total_fetched = 0
    total_processed = 0
    
    source_schema_name = row["raw_schema_name"]
    source_table_name = row["raw_table_name"]
    load_type = row["load_type"]
    target_schema_name = row["clean_schema_name"]
    target_table_name = row["clean_table_name"]
    target_columns = [col.strip() for col in row["column_name"].split(",")]
    load_key = row["load_key"]
    extract_key = row["extract_key"]
    source_name = row["source_name"]
    
    try:
        # Check if table exists, if not create it
        exists, columns = check_table_exists_and_get_columns(connection_silver_db, target_schema_name, target_table_name)
        if exists:
            sync_table_columns_with_target(connection_silver_db, target_schema_name, target_table_name, columns, target_columns)
        else:
            create_table_query = get_filtered_schema_for_create(connection_raw_db, source_table_name, target_columns, schema=source_schema_name)
            create_table_in_silver(connection_silver_db, target_schema_name, target_table_name, create_table_query, load_key)
        
        # Build where clause with upper bound
        where_clause = build_upper_bound_where_clause(
            extract_key_str=extract_key,
            upper_bound_date=upper_bound_time,
            dbtype=dbtype
        )
        logging.info(f"  Upper bound filter: {where_clause}")
                    
        # Truncate target table for full load
        truncate_query = f"""
            TRUNCATE TABLE "{target_schema_name}"."{target_table_name}" RESTART IDENTITY CASCADE;
        """
        with connection_silver_db.cursor() as cursor:
            cursor.execute(truncate_query)
            connection_silver_db.commit()
        
        conflict_columns = [col.strip() for col in load_key.split(",")] if load_key else []
        json_cols, array_cols = get_json_and_array_columns(
            connection_silver_db, target_schema_name, target_table_name
        )
        xml_cols = get_xml_columns(connection_silver_db, target_schema_name, target_table_name)   # ' ADDED
        
        new_columns_to_check = []
        batch_count = 0
        
        # Extract data from raw layer in batches with upper bound filter
        for batch in extract_source_table_data(
            row, connection_raw_db, batch_size=batch_size, where_clause=where_clause):
            
            batch_count += 1
            
            # Apply transformations
            if row.get('case_unification'):
                batch = apply_case_unification(batch, row['case_unification'], process_id, source_name, row['raw_table_name'], connection_silver_db)
            
            if row.get('is_datetime_formatting'):
                batch = apply_datetime_formatting(batch, row['is_datetime_formatting'], process_id, source_name, row['raw_table_name'], connection_silver_db)
            
            if row.get('is_json_xml_flatten'):
                batch = flatten_json_xml(batch, row['is_json_xml_flatten'])
            
            if row.get('is_add_indicator'):
                new_columns_to_check.extend(get_new_columns_from_config(row['is_add_indicator']))
                new_columns_to_check = list(set(new_columns_to_check))
                for col in new_columns_to_check:
                    if col not in batch.columns:
                        batch[col] = None

            if row.get('is_join_enrichment'):
                new_columns_to_check.extend(get_new_columns_from_config(row['is_join_enrichment']))
                new_columns_to_check = list(set(new_columns_to_check))
                for col in new_columns_to_check:
                    if col not in batch.columns:
                        batch[col] = None

            if row.get('is_country_code_unification'):
                batch = unify_country_code(batch, row['is_country_code_unification'])

            if row.get('is_null_handling') == '1':
                batch = handle_null_values(batch, process_id, source_name, row['raw_table_name'], connection_silver_db)

            if row.get('is_index_partition'):
                batch = index_partitioning(batch)

            if row.get('is_string_trimming') == '1':
                batch = trim_strings(batch)

            if row.get('is_currency_standardization'):
                batch = standardize_currency(batch, row['is_currency_standardization'].split(", "))

            if row.get('geographic_enrichment'):
                batch = enrich_geographic_data(batch, "latitude", "longitude")

            if row.get('base_currency') == 1:
                batch = standardize_base_currency(batch, row['column_name'].split(", "))

            if row.get('value_mapping_unification'):
                batch = apply_value_mapping_unification(batch, row['value_mapping_unification'])
            
            if row.get('is_duplicate_check') == '1':
                batch = remove_duplicates(batch.copy(), process_id, source_name, row["raw_table_name"], connection_silver_db)
                
            if row.get('is_post_2015_filter') == '1':
                batch = apply_post_2015_filter(batch.copy(), row['extract_key'], process_id, source_name, row['raw_table_name'], connection_silver_db)
            
            total_fetched += len(batch)
            # Filter columns to match target schema
            filtered_batch = batch[target_columns]
            filtered_batch["insertion_date"] = datetime.now()
            
            # Write the data to Silver Layer Table
            is_writing_success = write_batch_to_datalake(
                pg_conn=connection_silver_db,
                batch_df=filtered_batch,
                target_schema=target_schema_name,
                target_table=target_table_name,
                json_columns=json_cols,
                array_columns=array_cols,
                xml_columns=xml_cols,           # ' ADDED
                load_type=load_type,
                conflict_columns=conflict_columns
            )
            
            if is_writing_success:
                total_processed += len(filtered_batch)
            else:
                error_message = f"Batch {batch_count} failed to write to {target_schema_name}.{target_table_name}"
                final_status = "Failed"
                break
            
        # Determine final status
        if total_fetched == total_processed and total_fetched > 0:
            final_status = "Success"
            error_message = None
        elif total_processed > 0:
            final_status = "Partial Success"
            error_message = f"Processed {total_processed} out of {total_fetched} records"
        elif total_fetched == 0 and total_processed == 0:
            final_status = "Success"
        else:
            final_status = 'Failed'
            error_message = f"Row count mismatch: Fetched={total_fetched}, Processed={total_processed}"
    except Exception as e:
        error_message = str(e)
        final_status = 'Failed'
        connection_silver_db.rollback()
        raise
    finally:
        return total_fetched, total_processed, final_status, error_message


# ============================================================================
# MAIN ETL FUNCTION
# ============================================================================

def main(
    connection_details,
    source_name,
    table_type,
    etl_name,
    table_name=None,
    source_db_code=None,
    batch_size=100000,
    dbtype='postgres'
):
    """
    Main ETL function that handles both DELTA and FULL loads based on global batch audit table.
    """
    process_id = None    
    etl_end_time = None
    total_tables = 0
    successful_tables = 0
    failed_tables = 0
    connection_raw_db = None
    connection_silver_db = None
    batch_size = int(batch_size)
    overall_error_message = None
    batch_run_id = None
    run_type = None
    run_frequency = None
    etl_upper_bound_time = None
    
    logging.info("=" * 80)
    logging.info(f"ETL STARTED: {etl_name} | {datetime.now()}")
    logging.info("=" * 80)
    
    try:
        # Step 1: Connect to Silver DB
        connection_silver_db = initialize_database_connection(
            db_type="postgres", 
            **{**connection_details['pg_silver_connection']}
        )
        logging.info("Connected to Silver database")
        
        # Step 2: Read Global Batch Audit Information
        batch_run_id, run_start_datetime, etl_upper_bound_time, run_type, run_frequency = read_global_etl_batch_id(connection_silver_db)
        
        if not batch_run_id:
            error_msg = "No running batch found in global batch audit table"
            logging.error(error_msg)
            return {
                'status': 'Failed',
                'total_tables': 0,
                'successful_tables': 0,
                'failed_tables': 0,
                'message': error_msg
            }
        
        logging.info(f"Global Batch Information Retrieved:")
        logging.info(f"  - Batch Run ID: {batch_run_id}")
        logging.info(f"  - Run Type: {run_type}")
        logging.info(f"  - Run Frequency: {run_frequency}")
        logging.info(f"  - Upper Bound Time: {etl_upper_bound_time}")
        
        # Step 3: Connect to Raw DB
        connection_raw_db = initialize_database_connection(
            db_type="postgres", 
            **{**connection_details['pg_raw_connection']}
        )
        logging.info("Connected to Raw database")
        
        # Step 4: Initialize Master Audit
        etl_start_time = datetime.now()
        process_id = update_audit_master_tbl(
            connection_silver_db,
            batch_run_id=batch_run_id,
            etl_name=etl_name,
            etl_start_time=etl_start_time,
            etl_end_time=None,
            status="In-Progress",
            table_processed=0,
            table_succeeded=0,
            table_failed=0,
            error_msg=None,
            process_id=None,
            etl_schedule_time=etl_start_time
        )   
        
        logging.info(f"Process ID Created: {process_id}")
        logging.info("-" * 80)
        
        # Step 5: Read Config Table
        config_tbl = read_etl_config_table(
            connection_silver_db,
            source_name,
            table_type,
            run_frequency,
            run_type,
            table_name=table_name,
        )
        
        if config_tbl.empty:
            logging.warning("No tables found matching the criteria")
            
            update_audit_master_tbl(
                connection_silver_db,
                batch_run_id=batch_run_id,
                etl_name=etl_name,
                etl_start_time=None,
                etl_end_time=datetime.now(),
                status="Success",
                table_processed=0,
                table_succeeded=0,
                table_failed=0,
                error_msg="No tables found matching the criteria.",
                process_id=process_id,
            )
            return {
                'status': 'Success',
                'total_tables': 0,
                'successful_tables': 0,
                'failed_tables': 0,
                'message': 'No tables found matching the criteria.'
            }
        
        etl_metainfo_df = config_tbl.sort_values(by=["hierarchy_number", "load_sequence"])
        logging.info(f"Configuration loaded: {len(etl_metainfo_df)} tables to process")
        logging.info(f"Run Mode: {run_type.upper()}")
        logging.info("")
        
        # Step 6: Process Tables Based on Run Type
        for index, row in etl_metainfo_df.iterrows():
            total_tables += 1
            table_process_start_time = datetime.now()
            source_name_current = row["source_name"]
            source_schema = row["raw_schema_name"]
            source_table = row["raw_table_name"]
            destination_table = row["clean_table_name"]
            operation = row["load_type"]
            
            logging.info("-" * 80)
            logging.info(f"Processing [{total_tables}/{len(etl_metainfo_df)}]: {source_schema}.{source_table}")
            logging.info(f"  Mode: {run_type} | Operation: {operation}")
            
            try:
                # Insert initial audit record
                upsert_etl_audit_record(
                    connection_silver_db,
                    process_id,
                    batch_run_id,
                    source_name_current,
                    source_table,
                    destination_table,
                    operation,
                    time_from=run_start_datetime,
                    input_row=0,
                    status="In-Progress",
                    time_to=etl_upper_bound_time,
                    output_row=0,
                    error_code=None,
                    result=None,
                    etl_start_time=etl_start_time
                )
                
                # BIFURCATION: Delta vs Full Load
                if run_type.upper() == "DELTA":
                    logging.info(f"  Processing Type: INCREMENTAL/DELTA")
                    
                    last_processed_date_val = None
                    if operation.lower() in ("incremental", "append"):
                        last_processed_date_query = f"""
                            select
                                (MAX(sertal.time_to ) - interval '5 minutes') as last_processed_date
                            from
                                silver_etl_master.silver_etl_run_table_audit_log sertal 
                            join silver_etl_master.silver_etl_global_batch_audit segba 
                                on sertal.batch_run_id = segba.batch_run_id 
                            where
                                segba.load_type = 'DELTA'
                                and    operation != 'Post-Clean_Update'
                                and source_table_name = '{source_table}'
                                and operation_result = 'Success';
                        """
                        try:
                            result = pd.read_sql_query(last_processed_date_query, connection_silver_db)
                            if not result.empty and pd.notnull(result.iloc[0, 0]):
                                last_processed_date_val = result.iloc[0, 0]
#                                 last_processed_date_val = '2026-01-01 00:00:00.000' # --------------------------Debugging
                                logging.info(f"  Last Processed: {last_processed_date_val}")
                            else:
                                last_processed_date_val = datetime(2015, 1, 1, 0, 0, 0)
                                logging.info(f"  No previous run found, using: {last_processed_date_val}")
                        except Exception as e:
                            logging.warning(f"  Could not retrieve last processed date: {e}")
                            last_processed_date_val = etl_upper_bound_time - timedelta(days=30)
                            
                    elif operation.lower() == 'full':   
                        target_schema_name = row["clean_schema_name"]
                        target_table_name = row["clean_table_name"]
                        # Truncate target table for full load
                        truncate_query = f"""
                            TRUNCATE TABLE "{target_schema_name}"."{target_table_name}" RESTART IDENTITY CASCADE;
                        """
                        with connection_silver_db.cursor() as cursor:
                            cursor.execute(truncate_query)
                            connection_silver_db.commit()
                    
                    total_fetched, total_processed, final_status, error_message = process_table_delta(
                        connection_silver_db=connection_silver_db,
                        connection_raw_db=connection_raw_db,
                        row=row,
                        batch_size=batch_size,
                        process_id=process_id,
                        batch_run_id=batch_run_id,
                        last_processed_date_val=last_processed_date_val,
                        etl_start_datetime=etl_upper_bound_time,
                        dbtype=dbtype
                    )
                    
                elif run_type.upper() == "FULL":
                    logging.info(f"  Processing Type: FULL RELOAD with Upper Bound")
                    
                    total_fetched, total_processed, final_status, error_message = process_table_full(
                        connection_silver_db=connection_silver_db,
                        connection_raw_db=connection_raw_db,
                        process_id=process_id,
                        row=row,
                        upper_bound_time=etl_upper_bound_time,
                        batch_size=batch_size,
                        dbtype=dbtype
                    )
                    
                else:
                    error_message = f"Unidentified run type: {run_type}"
                    logging.error(f"  {error_message}")
                    failed_tables += 1
                    
                    upsert_etl_audit_record(
                        connection_silver_db,
                        process_id,
                        batch_run_id,
                        source_name_current,
                        source_table,
                        destination_table,
                        operation,
                        time_from=run_start_datetime,
                        input_row=0,
                        status="Failed",
                        time_to=etl_upper_bound_time,
                        output_row=0,
                        error_code=error_message,
                        result="Unsuccessful",
                    )
                    continue

                # Evaluate Results
                table_end_time = datetime.now()
                table_duration = table_end_time - table_process_start_time
                
                if final_status.lower() in ['success', 'partial success']:
                    successful_tables += 1
                    result_status = "Successful"
                    logging.info(f"  SUCCESS")
                    logging.info(f"    Fetched: {total_fetched:,} | Processed: {total_processed:,}")
                    logging.info(f"    Duration: {table_duration}")
                else:
                    failed_tables += 1
                    result_status = "Unsuccessful"
                    logging.error(f"  FAILED: {error_message}")
                    
                # Update final audit record
                upsert_etl_audit_record(
                    connection_silver_db,
                    process_id,
                    batch_run_id,
                    source_name_current,
                    source_table,
                    destination_table,
                    operation,
                    time_from=run_start_datetime,
                    input_row=total_fetched,
                    status=final_status,
                    time_to=etl_upper_bound_time,
                    output_row=total_processed,
                    error_code=error_message,
                    result=result_status,
                )
                                        
            except Exception as table_error:
                failed_tables += 1
                table_error_message = str(table_error)
                logging.error(f"  ERROR: {table_error_message}")
                logging.error(f"    Traceback: {traceback.format_exc()}")
                
                try:
                    upsert_etl_audit_record(
                        connection_silver_db,
                        process_id,
                        batch_run_id,
                        source_name_current,
                        source_table,
                        destination_table,
                        operation,
                        time_from=run_start_datetime,
                        input_row=0,
                        status="Failed",
                        time_to=etl_upper_bound_time,
                        output_row=0,
                        error_code=table_error_message,
                        result="Unsuccessful",
                    )
                except Exception as audit_error:
                    logging.error(f"  Failed to update audit record: {audit_error}")
        
        # Step 7: Final Summary and Audit Update
        etl_end_time = datetime.now()
        etl_duration = etl_end_time - etl_start_time
        
        if failed_tables == 0:
            overall_status = "Success"
            overall_error_message = None
        elif successful_tables > 0:
            overall_status = "Partial Success"
            overall_error_message = f"{failed_tables} out of {total_tables} tables failed"
        else:
            overall_status = "Failed"
            overall_error_message = "All tables failed to process"
        
        logging.info("")
        logging.info("=" * 80)
        logging.info(f"ETL SUMMARY: {overall_status}")
        logging.info(f"  Batch Run ID: {batch_run_id}")
        logging.info(f"  Run Type: {run_type}")
        logging.info(f"  Total Tables: {total_tables}")
        logging.info(f"  Successful: {successful_tables}")
        logging.info(f"  Failed: {failed_tables}")
        logging.info(f"  Duration: {etl_duration}")
        logging.info("=" * 80)
        
        # Update master audit table
        update_audit_master_tbl(
            connection_silver_db,
            batch_run_id=batch_run_id,
            etl_name=etl_name,
            etl_start_time=None,
            etl_end_time=etl_end_time,
            status=overall_status,
            table_processed=total_tables,
            table_succeeded=successful_tables,
            table_failed=failed_tables,
            error_msg=overall_error_message,
            process_id=process_id,
        )
        
        return {
            'status': overall_status,
            'batch_run_id': batch_run_id,
            'run_type': run_type,
            'total_tables': total_tables,
            'successful_tables': successful_tables,
            'failed_tables': failed_tables,
            'duration': str(etl_duration),
            'process_id': process_id,
            'message': overall_error_message
        }
        
    except Exception as main_error:
        etl_end_time = datetime.now()
        main_error_message = str(main_error)
        logging.error("=" * 80)
        logging.error(f"CRITICAL ERROR: {main_error_message}")
        logging.error(f"Traceback: {traceback.format_exc()}")
        logging.error("=" * 80)
        
        try:
            if process_id and batch_run_id:
                update_audit_master_tbl(
                    connection_silver_db,
                    batch_run_id=batch_run_id,
                    etl_name=etl_name,
                    etl_start_time=None,
                    etl_end_time=etl_end_time,
                    status="Failed",
                    table_processed=total_tables,
                    table_succeeded=successful_tables,
                    table_failed=failed_tables,
                    error_msg=main_error_message,
                    process_id=process_id,
                )
        except Exception as audit_error:
            logging.error(f"Failed to update master audit: {audit_error}")
        
        return {
            'status': 'Failed',
            'batch_run_id': batch_run_id,
            'run_type': run_type,
            'total_tables': total_tables,
            'successful_tables': successful_tables,
            'failed_tables': failed_tables,
            'process_id': process_id,
            'error': main_error_message
        }
        
    finally:
        try:
            if connection_raw_db:
                connection_raw_db.close()
                logging.info("Raw database connection closed")
            if connection_silver_db:
                connection_silver_db.close()
                logging.info("Silver database connection closed")
        except Exception as e:
            logging.error(f"Error closing connections: {e}")
        
        logging.info("=" * 80)
        logging.info(f"ETL FINISHED: {etl_name} | {datetime.now()}")
        logging.info("=" * 80)


# ============================================================================
# ENTRY POINT FUNCTION
# ============================================================================

def get_connection_details(
    pg_bronze_dbname,
    pg_silver_dbname,
    pg_dl_user,
    pg_dl_password,
    pg_dl_host,
    pg_dl_port,
    source_name,
    table_type,
    etl_name,
    table_name=None,
    source_db_code=None,
    batch_size=100000,
    dbtype='postgres'
):
    """
    Entry point function to configure connections and run the ETL process.
    """
    connection_details = {
        'pg_raw_connection': {
            'pg_dbname': pg_bronze_dbname,
            'pg_user': pg_dl_user,
            'pg_password': pg_dl_password,
            'pg_host': pg_dl_host,
            'pg_port': pg_dl_port
        },
        'pg_silver_connection': {
            'pg_dbname': pg_silver_dbname,
            'pg_user': pg_dl_user,
            'pg_password': pg_dl_password,
            'pg_host': pg_dl_host,
            'pg_port': pg_dl_port
        }
    }
    
    return main(
        connection_details,
        source_name,
        table_type,
        etl_name,
        table_name,
        source_db_code,
        batch_size,
        dbtype
    )

In [ ]:
# Data Lake Credentials
pg_dl_bronze_dbname = 'bronze_db'
pg_dl_silver_dbname='silver_db'
pg_dl_user = 'user'
pg_dl_password = 'password'
pg_dl_host = 'host'
pg_dl_port = 'port'

source_name = ('Travea',)
table_type = 'Master'

In [ ]:
tables = (
    "raw_tbl_ac_account",
    "raw_tbl_ac_chart_of_ac",
    "raw_tbl_ac_customer_data",
    "raw_tbl_ac_mgmt_pandl_glcode",
    "raw_tbl_ac_transaction",
    "raw_tbl_ba_documents",
    "raw_tbl_service",
    "raw_tbl_service_fare",
    "raw_tbl_ta_counter_staff",
    "raw_tbl_ta_service",
    "raw_tbl_ta_supp_doc_no"
)

In [ ]:
result = get_connection_details(
    pg_bronze_dbname=pg_dl_bronze_dbname,
    pg_silver_dbname=pg_dl_silver_dbname,
    pg_dl_user=pg_dl_user,
    pg_dl_password=pg_dl_password,
    pg_dl_host=pg_dl_host,
    pg_dl_port=pg_dl_port,
    source_name=source_name,
    table_type=table_type,
    etl_name='SILVER_ETL_DAILY_MASTER_RUN',
    table_name=tables,
    source_db_code=('UTTL-UAE',),
    batch_size=100000,
    dbtype='postgres',
)